# 04 — Pandas business tasks


> **Disclaimer:** Task descriptions were translated by the repository author using AI assistance (Cursor AI). This is **not** a professional translation. If anything is unclear, refer to code syntax, file names, and [HINTS.md](../../notes/HINTS.md).


In [ ]:
from pathlib import Path

ROOT = Path('../..').resolve()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'data' / 'output'
OUT.mkdir(parents=True, exist_ok=True)


## 📊 TASK: Regional Sales Audit

### 🎯 Business Context:
#### Sales leadership received a semi-annual export where each month is a separate column. This wide format hides temporal patterns, making it impossible to compare regional dynamics or identify whether revenue is driven by consistent performance or isolated spikes. You need to transform the data into a decision-ready matrix that highlights months contributing significantly to regional totals.

### 📋 Task:
#### Convert the wide table into long format: one row per product + region + month. ✅
#### Calculate each month's revenue share of the total annual revenue for that specific product-region pair. ✅
#### Filter to keep only months where the share exceeds 22% (the significance threshold for meaningful performance drivers). ✅
#### Reshape the filtered data into a summary matrix: rows = product, columns = region, values = total revenue from the filtered months. Fill missing combinations with 0.
#### Export the result to a CSV.


#### ✅ Allowed: melt, groupby + transform, column arithmetic, boolean filtering, pivot_table, to_csv.

#### ❌ Forbidden: merge, concat, apply, loops, manual row-wise calculations.

#### 📝 Expected Output: DataFrame with product as index, regions as columns, non-negative values, ready for BI import or executive slides.

In [ ]:
# Your solution here


📊 TASK 1: Customer 360 Cleanup (0:05–1:05)

💼 Business Brief: Marketing and Finance maintain separate customer databases. Overlapping IDs, duplicate records, and missing signup dates block campaign targeting and LTV calculation. You need a single clean master table to identify high-value customers by region.

📋 Non-Technical TZ
<!-- Load both files. Ensure customer IDs are treated as text, not numbers.  
Remove exact duplicate customer records.   -->
    Combine total invoices and maximum invoice amount per customer, then attach this to the customer list. Every customer must appear, even if they never bought.
    Fill missing signup dates with the median date of their region.
    Mark customers as High_Value if they have at least one paid invoice exceeding $200.
    Export.

    ⚙️ Technical Constraints:
    ✅ Allowed: read_csv(dtype, parse_dates), drop_duplicates(), groupby+agg(), merge(how='left'), fillna(), np.where(), to_csv()
    ❌ Forbidden: apply, loops, pivot, manual row iteration
    📤 Output: customer_master.csv | Columns: cust_id, region, signup_date, total_invoices, max_invoice_amount, is_high_value
    
    📝 Business Insight (2 bullets): Which region has the highest % of High_Value customers? How many customers have zero billing data?

In [ ]:
# Your solution here


# 📊 Task 2: User Engagement & Churn Risk Analysis

**📂 Input Data:** `user_events.csv`  
**🎯 Goal:** Identify users showing personal behavioral decay by comparing their recent session activity against their own historical baseline.

---

## 💼 Business Context
The Product team suspects that "Pro" users are disengaging before renewing their annual subscriptions. Raw logs show individual clicks, not sessions. We need to identify users whose recent activity has dropped significantly compared to their **own historical baseline** (not just the segment average). This allows Support to prioritize outreach to users showing *personal* decay, rather than just those who are naturally low-activity.

---

## 📋 Non-Technical Specifications

1.  **Load & Parse:** Load `user_events.csv`. Parse `event_date` into a datetime object and extract the month in `YYYY-MM` format.
2.  **Define Sessions:** A new session starts if the time gap between consecutive events for the same user exceeds **30 minutes**. 
    *   *Action:* Assign a unique `session_id` to each group of events. 
    *   *Hint:* Sort by `user_id` and `event_date`, use `.diff()` to find gaps, compare to `pd.Timedelta('30min')`, and use `.cumsum()` to create incremental IDs.
3.  **Aggregate to Session Level:** For each `session_id`, calculate:
    *   `session_duration`: The difference between the latest and earliest event time in that session.
    *   `event_count`: The number of events in that session.
4.  **Calculate User Baselines:** For each user, compute their `user_global_avg_duration` (the average of all their session durations across the entire dataset).
5.  **Identify Decay:** 
    *   First, calculate `monthly_avg_duration` for each user-month.
    *   Flag the user-month as `is_at_risk` if `monthly_avg_duration` is less than **70%** of that user's `user_global_avg_duration`.
6.  **Filter for Stability:** Keep only users who have been active in at least **3 different months** (to ensure the baseline is statistically stable).
7.  **Export:** Save the final user-month table.

---

## ⚙️ Technical Constraints

| ✅ Allowed | ❌ Forbidden |
| :--- | :--- |
| `pd.to_datetime()` | `apply()` with lambdas |
| `.sort_values()` | Explicit `for`/`while` loops |
| `.diff()` / `.gt()` / `.cumsum()` | `merge()` / `join()` (Use `transform` for broadcasting) |
| `groupby().agg()` | Manual session bounding logic |
| `groupby().transform()` | |
| Boolean filtering / `nunique()` | |
| `to_csv()` | |

**Rationale:** Using `transform` to broadcast global baselines back to monthly rows is more efficient and cleaner than merging. Session detection via `diff` and `cumsum` is a standard high-performance Pandas pattern.

---

## 📤 Expected Output Format

### `churn_risk_analysis.csv`
| user_id | month | total_sessions | monthly_avg_duration | user_global_avg_duration | is_at_risk |
| :--- | :--- | :--- | :--- | :--- | :--- |
| U001 | 2023-01 | 15 | 12.5 | 14.2 | False |
| U001 | 2023-02 | 8 | 9.1 | 14.2 | True |
| U002 | 2023-01 | 22 | 18.0 | 17.5 | False |

---

## 📝 Business Deliverable (Insights)

Provide 2 bullet points (4 sentences each) answering:

1.  **Risk Prevalence:** What percentage of "Pro" users are flagged `At_Risk` in the most recent month of the dataset? How does this compare to "Free" or "Enterprise" segments?
2.  **Driver Analysis:** Does the `is_at_risk` flag correlate more strongly with a drop in `total_sessions` (frequency) or a drop in `monthly_avg_duration` (depth)? Which metric should Support prioritize when reaching out?

Code
Reading files and parsing dates


In [ ]:
# Your solution here


# 📊 Task 3: Channel Concentration & Risk Diagnostic
  
**📂 Input Data:** `channel_sales_wide.csv`  
**🎯 Goal:** Build a diagnostic dataset that quantifies channel concentration risk and classifies every product-channel pair by its strategic role.

---

## 💼 Business Context
Leadership has two recurring concerns about our channel mix:
1.  **Concentration Risk:** Products with too much revenue from one channel are exposed to channel-specific shocks (e.g., platform fee hikes, account suspensions, partner churn).
2.  **Long-Tail Noise:** Channels that take operational overhead but contribute little revenue.

The current monthly review uses wide tables that hide these patterns. We need a diagnostic that flags both risks without losing information through arbitrary filtering.

---

## 📋 Non-Technical Specifications

1.  **Load & Inspect:** Load `channel_sales_wide.csv`. Confirm in a code comment that these are **annual** revenue figures.
2.  **Reshape to Long:** Convert the wide table to long format: one row per `(product, channel, revenue)`. 
    *   *Action:* Drop rows where `revenue` is 0 or `NaN`. 
    *   *Requirement:* Print the count of dropped rows (zero-revenue cells are meaningful—they indicate the channel doesn't carry the product).
3.  **Compute Three Share Metrics:** For every row, calculate three types of shares using `groupby().transform('sum')`:
    *   `share_of_product`: Revenue / Total Revenue for that **Product** (Channel Mix within Product).
    *   `share_of_channel`: Revenue / Total Revenue for that **Channel** (Product Mix within Channel).
    *   `share_of_total`: Revenue / **Grand Total** Revenue (Overall Importance to Company).
    *   *Why?* A pair can be small for the company but huge for a single product—that’s still concentration risk.
4.  **Calculate Concentration Risk (HHI):** Compute the Herfindahl-Hirschman Index for each product: 
    $$ HHI = \sum (share\_of\_product^2) $$
    *   *Action:* Use `groupby('product').transform()` to broadcast the HHI back to every row for that product.
    *   *Context:* HHI ranges 0–1; >0.4 is conventionally considered "concentrated."
5.  **Classify Risk:** Tag each row with a categorical `risk_class`:
    *   `"dominant"` if `share_of_product` > 0.35 (High concentration risk — surface this).
    *   `"core"` if `0.10 <= share_of_product <= 0.35`.
    *   `"long_tail"` if `share_of_product` < 0.10.
    *   *Note:* Do **not** filter rows out. All three classes must ship in the output.
6.  **Product-Level Summary:** Build a second DataFrame with one row per product containing:
    *   `total_revenue`
    *   `hhi_product`
    *   `top_channel` (the channel with the max revenue for that product)
    *   `top_channel_share`
    *   *Constraint:* Use `idxmax()` to find the top channel without using `apply`.
7.  **Export:** Save two files:
    *   `channel_pairs_long.csv` — The row-level table with shares and risk classes (BI-friendly, no information loss).
    *   `product_concentration_summary.csv` — One row per product, ranked by `hhi_product` descending.

---

## ⚙️ Technical Constraints

| ✅ Allowed | ❌ Forbidden |
| :--- | :--- |
| `melt()` | `merge()` / `join()` (Use `transform` for broadcasting) |
| `groupby().transform()` | `apply()` with lambdas (Vectorize or use named agg) |
| `groupby().agg()` (named aggregations) | Explicit `for`/`while` loops |
| `idxmax()` | `pivot_table()` for final output |
| Boolean masks / `pd.Categorical` | |
| `to_csv()` | |

**Rationale:** Long format preserves information; wide pivots lose the `risk_class` column and force `fill_value` ambiguity. The lesson is **"ship long to BI, let BI pivot."**

---

## 📤 Expected Output Formats

### 1. `channel_pairs_long.csv`
| product | channel | revenue | share_of_product | share_of_channel | share_of_total | hhi_product | risk_class |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| SKU_A | Organic | 5000.00 | 0.45 | 0.12 | 0.08 | 0.32 | dominant |
| SKU_A | Paid | 3000.00 | 0.27 | 0.09 | 0.05 | 0.32 | core |

### 2. `product_concentration_summary.csv`
| product | total_revenue | hhi_product | top_channel | top_channel_share |
| :--- | :--- | :--- | :--- | :--- |
| SKU_A | 11000.00 | 0.32 | Organic | 0.45 |
| SKU_B | 9500.00 | 0.41 | Partner | 0.52 |

---

## 📝 Business Deliverable (Insights)

Provide 3 bullet points (4 sentences each) answering:

1.  **Concentration Risks:** Which products have `hhi_product > 0.4` OR `top_channel_share > 0.5`? For each, name the dominant channel and quantify the exposure. Don't recommend action—just surface the exposure with numbers.
2.  **Long-Tail Analysis:** Where is the long tail genuine waste vs. strategic seeding? A long-tail pair on a new channel for a flagship product is a growth bet; on a declining product, it's overhead. State what additional data you'd need (e.g., channel age, CAC, product lifecycle stage) before recommending cuts.
3.  **Threshold Defense:** Defend or revise the 10% / 35% / 0.4 HHI thresholds. Pick one and either justify it from the data distribution (e.g., "35% is the 75th percentile of top-channel shares") or propose replacing it with a percentile-based rule. Arbitrary round numbers are a smell.

---

## 💡 Hint: Finding Top Channel Without `apply`

```python
# After computing shares, find the index of the max share for each product
idx_max = df.groupby('product')['share_of_product'].idxmax()

# Use these indices to pull the channel name and share
top_channels = df.loc[idx_max, ['product', 'channel', 'share_of_product']]

# Rename for clarity
top_channels.rename(columns={
    'channel': 'top_channel', 
    'share_of_product': 'top_channel_share'
}, inplace=True)

# Merge this back into your summary aggregation

In [ ]:
# After reading this file, it can be confirmed that these are annual revenue values.

# Making long table from a wide one (melting revenue channels into rows).
# Your solution here


In [ ]:
# Your solution here
